In [ ]:
from transformers import BertForMaskedLM
import sentencepiece as spm
import torch

MODEL_DIR = "../helabert_v2/bert_final_model"
TOKENIZER_MODEL = "../tokenizer/unigram_32000_0.9995.model"

# Load tokenizer
sp = spm.SentencePieceProcessor()
sp.load(TOKENIZER_MODEL)

# Load model
model = BertForMaskedLM.from_pretrained(MODEL_DIR)
model.eval()

sentence = "ශ්‍රී ලංකාවේ [MASK] අගනුවර කොළඹ වේ"

# Encode the sentence
input_ids = sp.encode(sentence, out_type=int)

# Debug: Check if [MASK] token exists
mask_id = sp.piece_to_id("[MASK]")
print(f"[MASK] token ID: {mask_id}")
print(f"Encoded tokens: {[sp.id_to_piece(id) for id in input_ids]}")

# Check if mask_id is in the encoded sequence
if mask_id not in input_ids:
    print("\n[MASK] token not found! Manually inserting it...")
    # Manually tokenize and insert mask
    parts = sentence.split("[MASK]")
    before = sp.encode(parts[0], out_type=int)
    after = sp.encode(parts[1], out_type=int)
    input_ids = before + [mask_id] + after

input_ids = torch.tensor([input_ids])

# Find mask position
mask_index = (input_ids == mask_id).nonzero(as_tuple=True)[1]

if len(mask_index) == 0:
    print("ERROR: [MASK] token still not found!")
else:
    with torch.no_grad():
        outputs = model(input_ids)
        logits = outputs.logits

    top_k = 5
    mask_logits = logits[0, mask_index]
    top_tokens = torch.topk(mask_logits, top_k, dim=-1)

    print(f"\nTop {top_k} predictions for [MASK]:")
    for token_id in top_tokens.indices[0]:
        print(sp.id_to_piece(token_id.item()))

In [ ]:
probs = torch.softmax(mask_logits, dim=-1)
top_probs = probs[0, top_tokens.indices[0]]

print(f"\nTop {top_k} predictions for [MASK]:")
for token_id, prob in zip(top_tokens.indices[0], top_probs):
    token = sp.id_to_piece(token_id.item())
    print(f"{token}: {prob.item():.4f}")